In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 57 samples and 9683 columns.
First few rows of the data:


,Overall survival time,Overall survival status,sample_title,Age,Sex,Grade,Pathological Stage,Muscular_infiltration,FLT3LG,SAP130,...,PPP4R2,EIF4G1,ADAM10,SLC10A3,TMEM141,RAB6A,PDPR,ST3GAL4,ZNF428,HIGD1A
0,377.0,1.0,tpm_CA1277013,58,Female,High,T4N0M0,MIBC,2.23,11.33,...,11.82,112.62,23.43,20.75,133.33,57.89,7.85,23.39,24.84,33.17
1,1518.0,0.0,tpm_CA1308866,50,male,High,T2N1M0,MIBC,17.90,21.74,...,16.25,146.06,43.80,70.66,93.95,104.52,13.68,105.12,29.61,92.48
2,946.0,0.0,tpm_CA1315330,63,male,High,T1NOMO,NMIBC,6.16,18.86,...,12.49,79.24,75.82,43.94,49.03,91.05,4.58,362.63,31.69,85.21
3,1258.0,0.0,tpm_CA1366278,53,male,High,T1NOMO,NMIBC,27.50,11.14,...,11.25,72.72,46.71,43.40,63.97,62.86,7.81,167.51,29.94,53.33
4,1213.0,0.0,tpm_CA1375590,46,male,High,T3bN1M0,MIBC,13.15,19.92,...,11.71,79.54,15.15,16.72,162.90,105.01,10.13,24.21,74.77,102.17


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for Overall survival time or Overall survival status: 1
Number of samples with complete data for both variables: 56
Number of censored samples (event_var == 0.0): 44
Number of events (event_var == 1.0): 12
Censored ratio (among complete): 44 / 56 = 0.786 (78.6%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE188715/description.json
